# Advanced Topics

This notebook covers advanced features: cross-analysis, statistical comparisons, and complex workflows.

In [29]:
import pandas as pd
import numpy as np
from scipy import stats
from pyMyriad import AnalysisTree, simple_table, gt_table
from pyMyriad.tabular import format_statistics, flatten, tabulate

In [34]:
# Create sample data
np.random.seed(42)
df = pd.DataFrame({
    "ID": range(1, 201),
    "Gender": np.random.choice(["M", "F"], 200),
    "Country": np.random.choice(["US", "UK", "Canada"], 200),
    "Age": np.random.randint(18, 70, 200),
    "Income": np.random.normal(50000, 15000, 200),
})

## Cross-Analysis with Reference Group

Use `cross_analyze_by()` to compare each group against a reference group.

In [ ]:
# Compare each country against US as reference
atree = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .cross_analyze_by(
        ref_lvl="US", # Specify reference level for Country
        mean_current=lambda df, ref_df: np.mean(df.Income),
        mean_reference=lambda df, ref_df: np.mean(ref_df.Income),
        difference=lambda df, ref_df: np.mean(df.Income) - np.mean(ref_df.Income),
    )

result = atree.run(df)

format_statistics(
    result,
    mean_current="{mean_current:.0f}",
    mean_reference="{mean_reference:.0f}",
    difference="{difference:.0f}",
    inplace=True
)

gt_table(result, by="df.Country", pivot_statistics=True)

GT(_tbl_data=  _Level_0 _Level_1 Canada_vs_US||difference Canada_vs_US||mean_current  \
0   Gender        F                      430                      53513   
1                 M                     3914                      49775   

  Canada_vs_US||mean_reference UK_vs_US||difference UK_vs_US||mean_current  \
0                        53083                -1376                  51707   
1                        45861                 7255                  53116   

  UK_vs_US||mean_reference  
0                    53083  
1                    45861  , _body=<great_tables._gt_data.Body object at 0x11b4f8080>, _boxhead=Boxhead([ColInfo(var='_Level_0', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_0', column_align='left', column_width=None), ColInfo(var='_Level_1', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_1', column_align='left', column_width=None), ColInfo(var='Canada_vs_US||difference', type=<ColInfoTypeEnum.default: 1>, column_label='difference', column_align='right', column_width=None), ColInfo(var='Canada_vs_US||mean_current', type=<ColInfoTypeEnum.default: 1>, column_label='mean_current', column_align='right', column_width=None), ColInfo(var='Canada_vs_US||mean_reference', type=<ColInfoTypeEnum.default: 1>, column_label='mean_reference', column_align='right', column_width=None), ColInfo(var='UK_vs_US||difference', type=<ColInfoTypeEnum.default: 1>, column_label='difference', column_align='right', column_width=None), ColInfo(var='UK_vs_US||mean_current', type=<ColInfoTypeEnum.default: 1>, column_label='mean_current', column_align='right', column_width=None), ColInfo(var='UK_vs_US||mean_reference', type=<ColInfoTypeEnum.default: 1>, column_label='mean_reference', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11b4fd8e0>, _spanners=Spanners([SpannerInfo(spanner_id='Hierarchy', spanner_level=0, spanner_label='Hierarchy', spanner_units=None, spanner_pattern=None, vars=['_Level_0', '_Level_1'], built=None), SpannerInfo(spanner_id='Canada_vs_US', spanner_level=0, spanner_label='Canada_vs_US', spanner_units=None, spanner_pattern=None, vars=['Canada_vs_US||difference', 'Canada_vs_US||mean_current', 'Canada_vs_US||mean_reference'], built=None), SpannerInfo(spanner_id='UK_vs_US', spanner_level=0, spanner_label='UK_vs_US', spanner_units=None, spanner_pattern=None, vars=['UK_vs_US||difference', 'UK_vs_US||mean_current', 'UK_vs_US||mean_reference'], built=None)]), _heading=Heading(title='Analysis Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11b4f9160>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11b4f87a0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11b4fa1b0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=T

## Cross-Analysis with Statistical Tests

Compute p-values comparing groups using t-tests.

In [44]:
# Function to perform t-test and return p-value
def ttest_pvalue(df, ref_df):
    if len(df) < 2 or len(ref_df) < 2:
        return np.nan
    t_stat, p_val = stats.ttest_ind(df.Income, ref_df.Income)
    return p_val

# Cross-analysis with p-values
atree2 = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .cross_analyze_by(
        ref_lvl="US",
        mean_diff=lambda df, ref_df: np.mean(df.Income) - np.mean(ref_df.Income),
        p_value=ttest_pvalue
    )

result2 = atree2.run(df)
format_statistics(
    result2,
    mean_diff="{mean_diff:.0f}",
    p_value="{p_value:.3f}",
    inplace=True
)

gt_table(result2, by="df.Country", pivot_statistics=True)

GT(_tbl_data=  _Level_0 _Level_1 Canada_vs_US||mean_diff Canada_vs_US||p_value  \
0   Gender        F                     430                 0.914   
1                 M                    3914                 0.287   

  UK_vs_US||mean_diff UK_vs_US||p_value  
0               -1376             0.676  
1                7255             0.054  , _body=<great_tables._gt_data.Body object at 0x11d716540>, _boxhead=Boxhead([ColInfo(var='_Level_0', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_0', column_align='left', column_width=None), ColInfo(var='_Level_1', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_1', column_align='left', column_width=None), ColInfo(var='Canada_vs_US||mean_diff', type=<ColInfoTypeEnum.default: 1>, column_label='mean_diff', column_align='right', column_width=None), ColInfo(var='Canada_vs_US||p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p_value', column_align='right', column_width=None), ColInfo(var='UK_vs_US||mean_diff', type=<ColInfoTypeEnum.default: 1>, column_label='mean_diff', column_align='right', column_width=None), ColInfo(var='UK_vs_US||p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p_value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11b505520>, _spanners=Spanners([SpannerInfo(spanner_id='Hierarchy', spanner_level=0, spanner_label='Hierarchy', spanner_units=None, spanner_pattern=None, vars=['_Level_0', '_Level_1'], built=None), SpannerInfo(spanner_id='Canada_vs_US', spanner_level=0, spanner_label='Canada_vs_US', spanner_units=None, spanner_pattern=None, vars=['Canada_vs_US||mean_diff', 'Canada_vs_US||p_value'], built=None), SpannerInfo(spanner_id='UK_vs_US', spanner_level=0, spanner_label='UK_vs_US', spanner_units=None, spanner_pattern=None, vars=['UK_vs_US||mean_diff', 'UK_vs_US||p_value'], built=None)]), _heading=Heading(title='Analysis Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11d53a180>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11d538b00>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11d539160>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A

## Complex Multi-Level Analysis with Tabulate

Create wide-format dataframe tables using `tabulate()`. The result is a dataframe that can be modified for other use.

In [25]:
# Create multi-level analysis
atree3 = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .analyze_by(
        mean=lambda df: np.mean(df.Income),
        sd=lambda df: np.std(df.Income),
        n=lambda df: len(df)
    )

result3 = atree3.run(df)
# Format statistics first
format_statistics(result3, mean_sd="{mean:.0f} ± {sd:.0f}", remove_original=True, inplace=True)

# Create wide-format table with tabulate
wide_table = tabulate(result3, pivot="df.Gender")
wide_table.head(10)


pivot_lvl,path_pivot,type,label,,F,M
0,root,level,NaN,NaN,None,None
1,root,root,NaN,None,NaN,NaN
2,root,split,NaN,None,NaN,NaN
3,root > df.Country,split,NaN,NaN,None,None
4,root > df.Country > Canada,level,NaN,NaN,None,None
5,root > df.Country > Canada > analysis,analysis,,NaN,"{'n': 31, 'mean_sd': '53513 ± 15584'}","{'n': 35, 'mean_sd': '49775 ± 16897'}"
6,root > df.Country > UK,level,NaN,NaN,None,None
7,root > df.Country > UK > analysis,analysis,,NaN,"{'n': 37, 'mean_sd': '51707 ± 11554'}","{'n': 24, 'mean_sd': '53116 ± 13582'}"
8,root > df.Country > US,level,NaN,NaN,None,None
9,root > df.Country > US > analysis,analysis,,NaN,"{'n': 32, 'mean_sd': '53083 ± 15227'}","{'n': 41, 'mean_sd': '45861 ± 14519'}"


## Advanced: Nested Cross-Analysis

Perform cross-analysis at multiple levels.

In [52]:
# Create age groups and perform nested cross-analysis
atree4 = AnalysisTree()\
    .split_by(
        label="Age Group",
        young=lambda df: df.Age < 40,
        older=lambda df: df.Age >= 40
    )\
    .split_by("df.Gender")\
    .cross_analyze_by(
        ref_lvl="M",
        mean_female=lambda df, ref_df: np.mean(df.Income) if len(df) > 0 else np.nan,
        mean_male=lambda df, ref_df: np.mean(ref_df.Income) if len(ref_df) > 0 else np.nan,
        gender_diff=lambda df, ref_df: np.mean(df.Income) - np.mean(ref_df.Income) if len(df) > 0 and len(ref_df) > 0 else np.nan
    )

result4 = atree4.run(df)
gt_table(result4)

GT(_tbl_data=     _Level_0 _Level_1 _Level_2 _Level_3    Statistic         Value
7   Age Group    young   Gender   F_vs_M  mean_female  52957.721603
7                                           mean_male  46480.776096
7                                         gender_diff   6476.945507
13               older                    mean_female  52518.293011
13                                          mean_male  49940.688239
13                                        gender_diff   2577.604772, _body=<great_tables._gt_data.Body object at 0x11b502930>, _boxhead=Boxhead([ColInfo(var='_Level_0', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_0', column_align='left', column_width=None), ColInfo(var='_Level_1', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_1', column_align='left', column_width=None), ColInfo(var='_Level_2', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_2', column_align='left', column_width=None), ColInfo(var='_Level_3', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_3', column_align='left', column_width=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.default: 1>, column_label='Statistic', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11b500da0>, _spanners=Spanners([SpannerInfo(spanner_id='Hierarchy', spanner_level=0, spanner_label='Hierarchy', spanner_units=None, spanner_pattern=None, vars=['_Level_0', '_Level_1', '_Level_2', '_Level_3'], built=None)]), _heading=Heading(title='Analysis Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11d2eb0e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11d2eb830>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11d2eb4d0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', va

## Complete Workflow: Analysis to Publication Table

A complete example from data to publication-ready output.

In [59]:
# 1. Define and run analysis
atree_final = AnalysisTree(denom="ID")\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .analyze_by(
        mean=lambda df: np.mean(df.Income),
        sd=lambda df: np.std(df.Income),
        n=lambda _N: _N[-1] / _N[0],
    )

result_final = atree_final.run(df)

# 2. Format statistics
format_statistics(result_final, mean_sd="{mean:.0f} ({sd:.0f})", remove_original=True, inplace=True)

# 3. Create publication table
pub_table = gt_table(result_final, by="df.Country", pivot_statistics=False)
pub_table

GT(_tbl_data=pivot_lvl _Level_0 _Level_1 Statistic         Canada             UK  \
0           Gender        F   mean_sd  53513 (15584)  51707 (11554)   
1                                   n          0.155          0.185   
2                         M   mean_sd  49775 (16897)  53116 (13582)   
3                                   n          0.175           0.12   

pivot_lvl             US  
0          53083 (15227)  
1                   0.16  
2          45861 (14519)  
3                  0.205  , _body=<great_tables._gt_data.Body object at 0x11b4f1400>, _boxhead=Boxhead([ColInfo(var='_Level_0', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_0', column_align='left', column_width=None), ColInfo(var='_Level_1', type=<ColInfoTypeEnum.default: 1>, column_label='_Level_1', column_align='left', column_width=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.default: 1>, column_label='Statistic', column_align='left', column_width=None), ColInfo(var='Canada', type=<ColInfoTypeEnum.default: 1>, column_label='Canada', column_align='right', column_width=None), ColInfo(var='UK', type=<ColInfoTypeEnum.default: 1>, column_label='UK', column_align='right', column_width=None), ColInfo(var='US', type=<ColInfoTypeEnum.default: 1>, column_label='US', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11d66cb90>, _spanners=Spanners([SpannerInfo(spanner_id='Hierarchy', spanner_level=0, spanner_label='Hierarchy', spanner_units=None, spanner_pattern=None, vars=['_Level_0', '_Level_1'], built=None)]), _heading=Heading(title='Analysis Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11d38e480>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11d38e9f0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11d38ed80>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='12px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_

## Denominator and Proportion Analysis

Use the `denom` parameter on `AnalysisTree` to count **unique entities** at each tree level
rather than rows.  This is important when observations are not uniquely identified by rows —
for example, a patient with two visits contributes 1 to the denominator, not 2.

When `denom` is set, `_N` is a cumulative list of unique counts:

| Index | Meaning |
|-------|---------|
| `_N[0]` | Unique count at the **root** (full dataset) |
| `_N[-1]` | Unique count at the **current node** |
| `_N[-2]` | Unique count at the **parent node** |

Lambdas receive `_N` automatically when their parameter is named `_N`:

| Lambda signature | Receives |
|------------------|---------|
| `lambda df: ...` | Current group DataFrame |
| `lambda _N: ...` | Denominator list `_N` |
| `lambda df, _N: ...` | Both DataFrame and list |
| `"_N[-1] / _N[0]"` (string) | `_N` injected into `eval` context |

In [58]:
# Patient data — patient C has two rows (two visits), so there are 3 unique patients
df_patients = pd.DataFrame({
    "PatientID": ["A", "B", "C", "C"],
    "Gender":    ["M", "F", "F", "F"],
    "Country":   ["US", "US", "UK", "UK"],
    "Income":    [48000, 52000, 55000, 58000],
})

# denom="PatientID" → _N counts unique PatientIDs, not rows
atree_denom = (AnalysisTree(denom="PatientID")
    .split_by("df.Gender")
    .analyze_by(
        mean_income   = lambda df:  np.mean(df.Income),
        n_patients    = lambda _N:  _N[-1],           # unique patients in this group
        prop_of_total = lambda _N:  _N[-1] / _N[0],  # proportion of all unique patients
    ))

result_denom = atree_denom.run(df_patients)

print("Root _N (total unique patients):", result_denom._N)
print("Female _N [total, female]:", result_denom["df.Gender"]["F"]._N)
print()
print(result_denom)

Root _N (total unique patients): [3]
Female _N [total, female]: [3, 2]

Data Tree
└─ Split: df.Gender
   ├─ F
   │  └─ analysis: 
   │     ├─ mean_income: 55000.0
   │     ├─ n_patients: 2
   │     └─ prop_of_total: 0.6666666666666666
   └─ M
      └─ analysis: 
         ├─ mean_income: 48000.0
         ├─ n_patients: 1
         └─ prop_of_total: 0.3333333333333333



### Multi-Level Proportions

With nested splits you can compute proportions relative to any ancestor by indexing `_N`.
`_N[-2]` is the parent level count, `_N[-1]` is the current level count.

In [11]:
atree_multilevel = (AnalysisTree(denom="PatientID")
    .split_by("df.Gender")
    .split_by("df.Country")
    .analyze_by(
        n                = lambda _N:     _N[-1],
        prop_vs_gender   = lambda _N:     _N[-1] / _N[-2],  # proportion within gender
        prop_vs_total    = lambda _N:     _N[-1] / _N[0],   # proportion vs full dataset
        # combine df and _N in one lambda
        row_vs_denom     = lambda df, _N: len(df) / _N[-1], # avg rows per unique patient
    ))

result_multilevel = atree_multilevel.run(df_patients)
simple_table(result_multilevel)

,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,Value
5,Gender,F,Country,UK,n,1
5,,,,,prop_vs_gender,0.5
5,,,,,prop_vs_total,0.333333
5,,,,,row_vs_denom,2.0
7,,,,US,n,1
7,,,,,prop_vs_gender,0.5
7,,,,,prop_vs_total,0.333333
7,,,,,row_vs_denom,1.0
11,,M,,,n,1
11,,,,,prop_vs_gender,1.0


### Multi-Column Denominator

Pass a list of columns to count unique **row combinations** — useful when entities are identified
by a composite key (e.g., PatientID + Visit).

In [12]:
df_visits = pd.DataFrame({
    "PatientID": ["A", "A", "B", "C", "C", "C"],
    "Visit":     [1,   2,   1,   1,   2,   3  ],
    "Gender":    ["M", "M", "F", "F", "F", "F"],
    "Score":     [10,  12,  8,   9,   11,  10 ],
})

# Count unique (PatientID, Visit) combinations — 6 combinations, not just 3 patients
atree_composite = (AnalysisTree(denom=["PatientID", "Visit"])
    .split_by("df.Gender")
    .analyze_by(
        mean_score  = lambda df:  np.mean(df.Score),
        n_visits    = lambda _N:  _N[-1],
        prop_visits = lambda _N:  _N[-1] / _N[0],
    ))

result_composite = atree_composite.run(df_visits)
print("Root _N (total unique patient-visits):", result_composite._N)
print(result_composite)

Root _N (total unique patient-visits): [6]
Data Tree
└─ Split: df.Gender
   ├─ F
   │  └─ analysis: 
   │     ├─ mean_score: 9.5
   │     ├─ n_visits: 4
   │     └─ prop_visits: 0.6666666666666666
   └─ M
      └─ analysis: 
         ├─ mean_score: 11.0
         ├─ n_visits: 2
         └─ prop_visits: 0.3333333333333333

